# Stage 2. Palm ROI Segmentation

Tidak ada mask ground truth untuk telapak tangan pada dataset ini sehingga stage ini tidak melatih model segmentasi baru seperti U-Net pada konjungtiva. Sebagai gantinya, region of interest dibangun dari convex hull landmark tangan yang diperhalus dengan thresholding warna kulit YCbCr lewat src.sites.palm.roi, lalu dinormalisasi iluminasi memakai CLAHE yang sama dengan situs lain agar konsisten lintas situs.

## Environment Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from configs import paths
from src.common import preprocess
from src.sites.palm import roi as palm_roi

output_dir = paths.outputs_dir("palm")
manifest = pd.read_csv(output_dir / "manifest.csv")
frame_qc = pd.read_csv(output_dir / "frame_qc.csv")
detected = frame_qc[frame_qc["detected"]].copy()
roi_dir = output_dir / "roi"
roi_dir.mkdir(parents=True, exist_ok=True)
print("frame terdeteksi", len(detected))

## Build ROI from Landmarks

Setiap frame terpilih diproses menjadi mask convex hull yang diperhalus warna kulit, lalu disimpan sebagai PNG RGBA dengan mask pada alpha channel agar langsung kompatibel dengan loader src.common.qc.load_roi tanpa penyesuaian tambahan.

In [ ]:
roi_records = []
for _, row in detected.iterrows():
    frame_rgb = cv2.cvtColor(cv2.imread(row["frame_path"]), cv2.COLOR_BGR2RGB)
    if row["method"] == "landmark":
        landmarks_px = np.load(row["landmarks_path"])
        result = palm_roi.palm_mask_from_landmarks(frame_rgb, landmarks_px)
    else:
        result = palm_roi.palm_mask_skin_only(frame_rgb)
    roi_path = roi_dir / f"{row['uid']}.png"
    palm_roi.save_roi_rgba(roi_path, result["rgb"], result["mask"])
    roi_records.append({
        "uid": row["uid"],
        "roi_path": str(roi_path),
        "roi_pixels": int(result["mask"].sum()),
        "method": row["method"],
    })

roi_qc = pd.DataFrame(roi_records)
print(roi_qc.groupby("method")["roi_pixels"].describe().round(1).to_string())

## Preview Segmentation Stages

Urutan raw frame, convex hull landmark, dan ROI akhir setelah irisan warna kulit, untuk verifikasi visual bahwa mask menutupi telapak tangan tanpa ikut menangkap latar belakang.

In [ ]:
def _preview_roi(row, ax_row):
    frame_rgb = cv2.cvtColor(cv2.imread(row["frame_path"]), cv2.COLOR_BGR2RGB)
    if row["method"] == "landmark":
        landmarks_px = np.load(row["landmarks_path"])
        result = palm_roi.palm_mask_from_landmarks(frame_rgb, landmarks_px)
        ax_row[0].scatter(landmarks_px[:, 0], landmarks_px[:, 1], s=10, c="red")
    else:
        result = palm_roi.palm_mask_skin_only(frame_rgb)
    ax_row[0].imshow(frame_rgb)
    ax_row[0].set_title(f"Raw Frame ({row['method']})")
    ax_row[1].imshow(result["mask"], cmap="gray")
    ax_row[1].set_title("Final ROI Mask")
    ax_row[2].imshow(result["rgb"])
    ax_row[2].set_title("Cropped ROI")
    for ax in ax_row:
        ax.axis("off")


landmark_sample = detected[detected["method"] == "landmark"].sample(1, random_state=7).iloc[0]
fallback_rows = detected[detected["method"] == "skin_fallback"]
samples = [landmark_sample]
if len(fallback_rows) > 0:
    samples.append(fallback_rows.sample(1, random_state=7).iloc[0])

fig, axes = plt.subplots(len(samples), 3, figsize=(13, 4 * len(samples)))
axes = np.atleast_2d(axes)
for ax_row, row in zip(axes, samples):
    _preview_roi(row, ax_row)
plt.tight_layout()
plt.show()

## Illumination Normalization Check

CLAHE pada channel V dijalankan lewat pipeline generik src.common.preprocess yang sama dengan situs konjungtiva, memverifikasi bahwa file ROI RGBA hasil stage ini terbaca benar oleh loader bersama.

In [ ]:
normalized = preprocess.normalize_roi(roi_qc.iloc[0]["roi_path"])
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(normalized["rgb"])
axes[0].set_title("Normalized ROI")
axes[1].imshow(normalized["valid_mask"], cmap="gray")
axes[1].set_title("Valid Pixel Mask")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Merge ROI into Manifest

Manifest diperbarui dengan roi_path hasil segmentasi. Baris tanpa ROI (video gagal unduh atau tangan tidak terdeteksi) tetap ada di manifest namun roi_path kosong, sehingga stage berikutnya dapat memfilter otomatis lewat notnull.

In [ ]:
manifest = manifest.drop(columns=["roi_path"]).merge(roi_qc[["uid", "roi_path"]], on="uid", how="left")
manifest["roi_path"] = manifest["roi_path"].fillna("")
manifest["roi_precropped"] = manifest["roi_path"] != ""
usable = manifest["roi_precropped"].sum()
print(f"sampel dengan ROI siap pakai: {usable} dari {len(manifest)}")

## Save Updated Manifest

In [ ]:
manifest.to_csv(output_dir / "manifest.csv", index=False)
roi_qc.to_csv(output_dir / "roi_qc.csv", index=False)
print("manifest dan roi qc tersimpan di", output_dir)